# 04 — Federated FedAvg baseline (Phase 3)

Simulated hospitals (Dirichlet ward-mixture) train together via **FedAvg** (Flower).
Compares **pooled** (Phase 1) vs **local-only** (each hospital alone) vs **FedAvg**.

**To get new code after a `git pull`:** just re-run cell 1 (pull) then the run cell —
cells 4/5 force-reload `amr_fed` from disk, so **no kernel restart is needed**.
All simulated on ONE machine — no second computer needed.

In [ ]:
# 1) Get the code + deps
!git clone -b phase3-federated https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout phase3-federated && git pull)
!pip install -q torch_geometric 'flwr[simulation]'

In [ ]:
# 2) Point at the data (mount Drive, set ARMD_DIR before importing amr_fed)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder

In [ ]:
# 3) Sanity: data resolves
import sys
sys.path.insert(0, '/content/Capstone-amr-fed/src')
from amr_fed import config
from pathlib import Path
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong — fix cell 2 and re-run.'

In [ ]:
# 4) Run FedAvg at alpha=0.5 (5 hospitals). Prints local-only vs FedAvg vs pooled.
# Force-reload amr_fed from disk so a `git pull` takes effect WITHOUT a kernel restart.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]

from amr_fed.federated.run import run_fedavg
res = run_fedavg(alpha=0.5, n_clients=5, rounds=10, local_epochs=6)
print(res)

In [ ]:
# 5) MULTI-SEED alpha sweep — mean +/- std to denoise partition + training noise.
# 3 seeds x 3 alphas = 9 FedAvg runs, ~20-40 min. Cohort loaded once and reused.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame

df = load_cohort_frame()
summaries = [run_multiseed(alpha=a, seeds=(42, 43, 44), df=df) for a in (0.1, 0.5, 1.0)]

print("\n=== ALPHA SWEEP (multi-seed, mean +/- std) ===")
for s in summaries:
    print(f"alpha={s['alpha']}: local {s['local_only'][0]}+/-{s['local_only'][1]} | "
          f"FedAvg-best {s['fedavg_best'][0]}+/-{s['fedavg_best'][1]} | "
          f"gain {s['gain_mean']}+/-{s['gain_std']}")

In [ ]:
# 6) NON-IID SPLIT #1: label-Dirichlet on RESISTANT-RATE (the axis FedAvg struggles with).
# Splits patients so hospitals differ in their resistant/susceptible mix. Sweeps beta
# (small = strong label skew). Expect local-only to DROP and the FedAvg gain + worst-
# hospital gain to GROW vs the ward split -- i.e. more room for the topology-aware method.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import label_dirichlet

df = load_cohort_frame()
for beta in (0.1, 0.5):
    run_multiseed(seeds=(42, 43, 44), df=df, label=f"label-dir beta={beta}",
                  partition_fn=lambda d, s, b=beta: label_dirichlet(d, n_clients=5, beta=b, seed=s))

In [ ]:
# 7) NON-IID SPLIT #3: natural SPECIMEN split (urine / respiratory / blood / other).
# One hospital per specimen source -- clinically defensible, and resistance varies a lot
# by source (EDA: urine ~0.18 vs resp ~0.29), so this is a real label + topology skew.
# n_clients is derived from the split (3-4 hospitals); seeds vary only training noise.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import specimen_baseline

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="specimen",
              partition_fn=lambda d, s: specimen_baseline(d))

In [ ]:
# 8) NON-IID SPLIT #4: ORGANISM-community split -- each hospital sees a DIFFERENT set of
# bugs (maximal structural/topology heterogeneity; the tightest fit for topology-aware
# aggregation). Organisms are greedily packed into 5 balanced hospitals. Expect the
# strongest FedAvg + worst-hospital gains of all the splits.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import organism_community

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="organism-community",
              partition_fn=lambda d, s: organism_community(d, n_clients=5, seed=s))

In [ ]:
# 9) HARD SPLIT #1: TOPOLOGY quadrant — crossed homophily × degree, purity=0.0.
# 8 hospitals = 4 corners × 2 buckets. Revised protocol (v2): wider model (128),
# more budget (8r×4e=32 epochs) so pooled isn't under-trained.
# v1 (hidden=64, 6r×3e): FedAvg 0.7036 beat pooled 0.6881 — no headroom.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from functools import partial
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import topology_split

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="topology-corners",
              partition_fn=partial(topology_split, purity=0.0, decorrelate=False),
              n_clients=8, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)

In [ ]:
# 10) HARD SPLIT #2: LOUVAIN communities — FedGTA-style topology split.
# Hospitals built from Louvain community detection on the organism–antibiotic
# test graph, greedily packed. Fewer communities than n_clients → run_fedavg self-heals.
# v1 (hidden=64): 3 hospitals (73%/21%/5%), FedAvg 0.6976 beat pooled 0.6596, BUT
# worst hospital got WORSE with FedAvg (0.6722→0.6614) — right shape, pooled too weak.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from amr_fed.federated.run import run_multiseed
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import louvain_split

df = load_cohort_frame()
run_multiseed(seeds=(42, 43, 44), df=df, label="louvain-communities",
              partition_fn=louvain_split, n_clients=5, rounds=8,
              local_epochs=4, local_only_epochs=32, hidden=128)

In [ ]:
# 11) HEADROOM GATE — the acceptance test for topology_split: does FedAvg trail pooled?
#   PASS = FedAvg trails pooled by >=0.02 (worst) or >=0.01 (mean)
#          AND pooled worst-hospital >= 0.60 (pooled still strong)
#          AND FedAvg beats local-only (it helps vs training alone)
# v1 (hidden=64): FAIL — FedAvg beat pooled by +0.015. v2: wider model (128) + more budget.
import sys
for _m in [m for m in list(sys.modules) if m.startswith('amr_fed')]:
    del sys.modules[_m]
from functools import partial
from amr_fed.federated.run import headroom_gate
from amr_fed.data_loader import load_cohort_frame
from amr_fed.partition import topology_split

df = load_cohort_frame()
res = headroom_gate(partition_fn=partial(topology_split, purity=0.0),
                    n_clients=8, rounds=8, local_epochs=4, hidden=128,
                    df=df, label="topology-corners")
# If FAIL:
#   fed_helps fails -> raise purity (0.2) / rounds (10) / local_epochs (6)
#   gap too small   -> lower purity to 0.0, try hidden=64, or rounds=6
#   pooled weak     -> widen hidden to 256, reduce n_clients to 4
# When PASS, record winning config in docs/2026-08-09-headroom-calibration.md